## First we generate the kinematics from our symbolic math in rrur_kinematics

In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
from rrur_kinematics import *
from visualize_rrur import visualize_robot

In [2]:
from sympy.printing.pycode import pycode
# Helper to generate function strings
def generate_function_string(name, args, expr, module="jnp"):
    """Generates a Python function as a string for a given SymPy expression."""
    arg_str = ", ".join(map(str, args))
    
    # Generate code for each element of the matrix if it's a matrix
    if expr.is_Matrix:
        lines = []
        for i in range(expr.rows):
            for j in range(expr.cols):
                lines.append(f"    m{i}{j} = {pycode(expr[i, j])}")
        
        matrix_repr = f"{module}.array([\n"
        for i in range(expr.rows):
            row_items = [f"m{i}{j}" for j in range(expr.cols)]
            matrix_repr += f"        [{', '.join(row_items)}],\n"
        matrix_repr += f"    ], dtype=float).ravel()"
        
        body = "\n".join(lines) + f"\n    return {matrix_repr}"
    else:
        body = f"    return {pycode(expr)}"

    return f"def {name}({arg_str}):\n    # Note: uses {module} for math functions like sin, cos\n{body}\n"

# Define arguments for each function based on symbolic dependencies
args_E1_chain = ('theta_11', 'theta_12', 'gamma_11', 'gamma_12')
args_E2_chain = ('theta_21', 'theta_22', 'gamma_21', 'gamma_22')
args_E3_chain = ('theta_31', 'theta_32', 'gamma_31', 'gamma_32')
args_Q1 = ('theta_11',)
args_Q2 = ('theta_21',)
args_Q3 = ('theta_31',)
args_U1 = ('theta_11', 'theta_12')
args_U2 = ('theta_21', 'theta_22')
args_U3 = ('theta_31', 'theta_32')
args_ee = ('alpha', 'beta', 'gamma', 'z')

# Generate function strings
E1_chain_func_str = generate_function_string('E1_chain_func', args_E1_chain, E1_chain_geo)
E2_chain_func_str = generate_function_string('E2_chain_func', args_E2_chain, E2_chain_geo)
E3_chain_func_str = generate_function_string('E3_chain_func', args_E3_chain, E3_chain_geo)

Q1_func_str = generate_function_string('Q1_func', args_Q1, Q1_geo)
Q2_func_str = generate_function_string('Q2_func', args_Q2, Q2_geo)
Q3_func_str = generate_function_string('Q3_func', args_Q3, Q3_geo)

U1_func_str = generate_function_string('U1_func', args_U1, U1_geo)
U2_func_str = generate_function_string('U2_func', args_U2, U2_geo)
U3_func_str = generate_function_string('U3_func', args_U3, U3_geo)

E1_ee_func_str = generate_function_string('E1_ee_func', args_ee, E1_ee_geo)
E2_ee_func_str = generate_function_string('E2_ee_func', args_ee, E2_ee_geo)
E3_ee_func_str = generate_function_string('E3_ee_func', args_ee, E3_ee_geo)
A_func_str = generate_function_string('A_func', args_ee, A_geo)

S1_func_str = generate_function_string('S1_func', [], S1_geo)
S2_func_str = generate_function_string('S2_func', [], S2_geo)
S3_func_str = generate_function_string('S3_func', [], S3_geo)

# You can now print these strings or write them to a file
# For example, to write to a file named 'generated_kinematics.py':
with open('generated_kinematics.py', 'w') as f:
    f.write("# This file is auto-generated. Do not edit manually.\n")
    f.write("import os\n")
    f.write("os.environ['JAX_PLATFORMS'] = 'cpu'\n")
    f.write("import jax\n")
    f.write("import optax\n")
    f.write("import jax.numpy as jnp\n\n")

    # List of all generated function strings
    all_func_strs = [
        E1_chain_func_str, E2_chain_func_str, E3_chain_func_str,
        Q1_func_str, Q2_func_str, Q3_func_str,
        E1_ee_func_str, E2_ee_func_str, E3_ee_func_str,
        A_func_str, S1_func_str, S2_func_str, S3_func_str,
        U1_func_str, U2_func_str, U3_func_str
    ]

    for func_str in all_func_strs:
        # Replace math and numpy with np (jax.numpy) and add the jax.jit decorator
        modified_str = func_str.replace('math.', 'jnp.').replace('numpy.', 'jnp.')
        jit_compiled_str = "@jax.jit\n" + modified_str
        f.write(jit_compiled_str + "\n")
    
    f.write(f"p1 = {p1}\n")
    f.write(f"p2 = {p2}\n")
    f.write(f"p3 = {p3}\n")
    
print("Generated JAX-compatible code has been written to generated_kinematics.py")


Generated JAX-compatible code has been written to generated_kinematics.py


## Develop FK and IK numeric solving

In [3]:
from generated_kinematics import *
import timeit

In [4]:
import jax
import jax.numpy as jnp
import optax

In [5]:
# Constants for the squared lengths of the passive rods
p1_squared = p1**2
p2_squared = p2**2
p3_squared = p3**2

@jax.jit
def objective(ee_params, joint_params, passive_params):
    """
    Kinematics Objective Function.
    Calculates the error vector for a given end-effector and joint configuration.
    
    Args:
        ee_params (jnp.array): An array containing the end-effector parameters 
                               [alpha, beta, gamma, z].
        joint_params (jnp.array): An array containing all 12 joint angles.

    Returns:
        float: The sum of squared errors for the virtual rods and the end-effector chains.
    """
    alpha, beta, gamma, z = ee_params
    theta_11, theta_21, theta_31, theta_22 = joint_params
    theta_12, theta_32, gamma_11, gamma_12, gamma_21, gamma_22, gamma_31, gamma_32 = passive_params

    # Calculate kinematic quantities based on joint and ee parameters
    U1_val = U1_func(theta_11, theta_12)
    U2_val = U2_func(theta_21, theta_22)
    U3_val = U3_func(theta_31, theta_32)

    E1_chain_val = E1_chain_func(theta_11, theta_12, gamma_11, gamma_12)
    E2_chain_val = E2_chain_func(theta_21, theta_22, gamma_21, gamma_22)
    E3_chain_val = E3_chain_func(theta_31, theta_32, gamma_31, gamma_32)

    A_val = A_func(alpha, beta, gamma, z)
    E1_ee_val = E1_ee_func(alpha, beta, gamma, z)
    E2_ee_val = E2_ee_func(alpha, beta, gamma, z)
    E3_ee_val = E3_ee_func(alpha, beta, gamma, z)

    # Calculate squared distance errors for the passive rods
    p1_squared_calc = jnp.sum((U1_val - A_val)**2)
    p2_squared_calc = jnp.sum((U2_val - A_val)**2)
    p3_squared_calc = jnp.sum((U3_val - A_val)**2)

    # Assemble the 12-element output error_squared vector
    output = jnp.zeros(12)
    output = output.at[0].set(p1_squared_calc - p1_squared)
    output = output.at[1].set(p2_squared_calc - p2_squared)
    output = output.at[2].set(p3_squared_calc - p3_squared)
    output = output.at[3:6].set(E1_chain_val - E1_ee_val)
    output = output.at[6:9].set(E2_chain_val - E2_ee_val)
    output = output.at[9:12].set(E3_chain_val - E3_ee_val)

    return jnp.sum(jnp.square(output))


# Pre-compile gradients for optimization
# Gradient for IK: objective w.r.t. joint and passive parameters
ik_objective_grad = jax.jit(jax.grad(lambda p, e: objective(e, p[:4], p[4:]), argnums=0))

# Gradient for FK: objective w.r.t. end-effector and passive parameters
fk_objective_grad = jax.jit(jax.grad(lambda p, j: objective(p[:4], j, p[4:]), argnums=0))

ik_x0 = jnp.array([jnp.deg2rad(30), jnp.deg2rad(130), jnp.deg2rad(150), jnp.deg2rad(270), jnp.deg2rad(90), jnp.deg2rad(270), jnp.deg2rad(90), 0, jnp.deg2rad(90), 0, jnp.deg2rad(90), 0])
fk_x0 = jnp.array([
    jnp.deg2rad(0),                # alpha (rad)
    jnp.deg2rad(0),                # beta (rad)
    jnp.deg2rad(0),                # gamma (rad)
    0.18,               # z (m)
    jnp.deg2rad(120),   # theta_12 (rad)
    jnp.deg2rad(240),   # theta_32 (rad)
    jnp.deg2rad(90),    # gamma_11 (rad)
    0.0,                # gamma_12 (rad)
    jnp.deg2rad(90),    # gamma_21 (rad)
    0.0,                # gamma_22 (rad)
    jnp.deg2rad(90),    # gamma_31 (rad)
    0.0                 # gamma_32 (rad)
])

@jax.jit
def inverse_kinematics(ee_params, x0=ik_x0, learning_rate=3e-1, max_iter=75, tol=1e-6):
    """
    Inverse kinematics solver using optax Adam optimizer.
    """
    optimizer = optax.adam(learning_rate)
    opt_state = optimizer.init(x0)
    params = x0

    def step(params, opt_state):
        grads = ik_objective_grad(params, ee_params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state

    for _ in range(max_iter):
        params, opt_state = step(params, opt_state)

    loss = objective(ee_params, params[:4], params[4:])
    return params, loss

@jax.jit
def forward_kinematics(joint_params, x0=fk_x0, learning_rate=3e-2, max_iter=75, tol=1e-6):
    """
    Forward kinematics solver using optax Adam optimizer.
    """
    optimizer = optax.adam(learning_rate)
    opt_state = optimizer.init(x0)
    params = x0

    def step(params, opt_state):
        grads = fk_objective_grad(params, joint_params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state

    for _ in range(max_iter):
        params, opt_state = step(params, opt_state)

    loss = objective(params[:4], joint_params, params[4:])
    return params, loss


In [6]:
import re
from IPython import get_ipython

# Get the code of the previous cell (cell index 4)
cell_code = In[5]
with open('generated_kinematics.py', 'a') as f:
    f.write('\n\n' + cell_code + '\n')
print("Appended previous cell's code to generated_kinematics.py")


Appended previous cell's code to generated_kinematics.py


In [212]:
# joint angle initial guess
ik_x0 = jnp.array([jnp.deg2rad(30), jnp.deg2rad(130), jnp.deg2rad(150), jnp.deg2rad(270), jnp.deg2rad(90), jnp.deg2rad(270), jnp.deg2rad(90), 0, jnp.deg2rad(90), 0, jnp.deg2rad(90), 0])

pose = jnp.array([jnp.deg2rad(0), jnp.deg2rad(0), jnp.deg2rad(0), 0.15])  # [alpha, beta, gamma, z]

In [213]:
# Print the device JAX is using
print(f"JAX is using: {jax.devices()}")

JAX is using: [CpuDevice(id=0)]


In [203]:
# Run Inverse Kinematics
num_runs = 1000  # You can change this value as needed

print(f"Running Inverse Kinematics {num_runs} times for timing...")
start_time = timeit.default_timer()
for _ in range(num_runs):
    ik_joint_params, loss = inverse_kinematics(pose, x0=ik_x0)
ik_joint_params.block_until_ready() # Block until the last computation is finished
end_time = timeit.default_timer()

total_time = end_time - start_time
print(f"Total IK execution time for {num_runs} runs: {total_time:.4f} seconds")
print(f"Average execution time per run: {total_time / num_runs:.6f} seconds")

print("Resulting Joint Parameters (degrees):", jnp.rad2deg(ik_joint_params[:4]))
print("Resulting Passive Parameters (degrees):", jnp.rad2deg(ik_joint_params[4:]))
print(f"Final loss: {loss:.6f}")

visualize_robot(*pose, *ik_joint_params)

Running Inverse Kinematics 1000 times for timing...
Total IK execution time for 1000 runs: 0.6076 seconds
Average execution time per run: 0.000608 seconds
Resulting Joint Parameters (degrees): [ 18.761078 149.2135   161.2389   254.80838 ]
Resulting Passive Parameters (degrees): [111.60468  248.39532   90.04559   41.834026  89.99949   51.09525
  90.0456    41.834026]
Final loss: 0.000006


In [215]:
# Run Forward Kinematics
target_joint_angles = jnp.array([jnp.deg2rad(30), jnp.deg2rad(150), jnp.deg2rad(150), jnp.deg2rad(270)])
num_runs = 1000

# The num_runs variable is defined in the cell above
print(f"Running Forward Kinematics {num_runs} times for timing...")
start_time = timeit.default_timer()
for _ in range(num_runs):
    fk_params, loss = forward_kinematics(target_joint_angles,x0=fk_x0, learning_rate=0.3)
fk_params.block_until_ready() # Block until the last computation is finished
end_time = timeit.default_timer()

total_time = end_time - start_time
print(f"Total FK execution time for {num_runs} runs: {total_time:.4f} seconds")
print(f"Average execution time per run: {total_time / num_runs:.6f} seconds")

# Extract results
fk_pose = fk_params[:4]
fk_passive_params = fk_params[4:]

#fk_pose = fk_x0[:4]
#fk_passive_params = fk_x0[4:]

# Print results
print("Resulting Pose (alpha(deg), beta(deg), gamma(deg), z(m)):", jnp.rad2deg(fk_pose[:3]), fk_pose[3])
print("Resulting Passive Parameters (degrees):", jnp.rad2deg(fk_passive_params))
print(f"Final loss: {loss:.6f}")

visualize_robot(*fk_pose, *target_joint_angles, *fk_passive_params)

Running Forward Kinematics 1000 times for timing...
Total FK execution time for 1000 runs: 1.0122 seconds
Average execution time per run: 0.001012 seconds
Resulting Pose (alpha(deg), beta(deg), gamma(deg), z(m)): [ 28.537418   1.017905 -17.4383  ] 0.116636686
Resulting Passive Parameters (degrees): [104.963524 249.0763    34.00248   40.579624  46.22322   66.67782
  72.13193   32.441162]
Final loss: 0.000165
